# Paso 4 — Hyperparameter Tuning

En este notebook optimizamos el **Random Forest**, que en el Notebook 3 presentó el mejor desempeño entre los modelos evaluados, aunque también mostró una brecha importante entre entrenamiento y validación.

El objetivo no es maximizar el resultado sobre Train. Buscamos un modelo que:

- mejore o mantenga el **F1 Macro**;
- generalice mejor sobre datos no vistos;
- reduzca, cuando sea posible, la brecha entre Train y validación/Test;
- conserve un pipeline reproducible para las etapas posteriores.

> **Regla profesional:** el conjunto Test permanece aislado durante todo el tuning. Los hiperparámetros se seleccionan únicamente usando Cross Validation sobre Train.

## Flujo del notebook

```text
Notebook 3
   ↓
Random Forest candidato
   ↓
Train / Test
   ↓
Pipeline + SMOTE
   ↓
RandomizedSearchCV
   ↓
Mejor combinación de hiperparámetros
   ↓
Reentrenamiento sobre TRAIN completo
   ↓
Cross Validation de confirmación
   ↓
Evaluación FINAL sobre TEST
   ↓
Análisis de generalización
   ↓
Guardar modelo optimizado
```

### ¿Por qué RandomizedSearchCV?

Probar todas las combinaciones posibles puede ser costoso. `RandomizedSearchCV` explora un número controlado de configuraciones y permite obtener una buena solución en menor tiempo.

Para este laboratorio utilizaremos una búsqueda compacta orientada a completar el flujo End-to-End. Se reducen deliberadamente los folds e iteraciones para ejecutar el laboratorio rápidamente en un Compute pequeño.

## 01. Librerías

Utilizamos `scikit-learn` para el modelo y la validación, `imbalanced-learn` para SMOTE dentro del Pipeline y `joblib` para persistir el modelo.

No actualizamos las versiones del entorno Azure ML. Se utilizan las librerías que ya fueron validadas en el Compute.

In [17]:
# ==========================================
# 01. Librerías
# ==========================================

import os
import time
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RandomizedSearchCV,
    cross_validate
)

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

print("Librerías cargadas correctamente.")
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("sklearn:", __import__("sklearn").__version__)
print("imblearn:", __import__("imblearn").__version__)

Librerías cargadas correctamente.
numpy: 1.23.5
pandas: 1.5.3
sklearn: 1.5.1
imblearn: 0.12.4


## 02. Configuración

Mantenemos la misma definición de datos utilizada en el Notebook 3 para que la comparación sea consistente.

El Test Set se vuelve a separar con la misma semilla y estratificación. **No se utiliza para elegir hiperparámetros.**

In [18]:
# ==========================================
# 02. Configuración
# ==========================================

INPUT_PATH = "../data/output_data/featureengineering_files.parquet"
MODEL_DIR = "../models"

os.makedirs(MODEL_DIR, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.30

# CV rápida para tuning.
# El Test permanece completamente aislado.
N_SPLITS_TUNING = 2
N_ITER_SEARCH = 3

TARGET = "label_e"

REMOVE_COLUMNS = [
    "machineID",
    "dt_truncated",
    "datetime",
    "failure",
    "model",
    TARGET
]

print("Dataset:", INPUT_PATH)
print("Modelo a optimizar: Random Forest")
print("Métrica principal: F1 Macro")
print("Iteraciones RandomizedSearchCV:", N_ITER_SEARCH)

Dataset: ../data/output_data/featureengineering_files.parquet
Modelo a optimizar: Random Forest
Métrica principal: F1 Macro
Iteraciones RandomizedSearchCV: 3


## 03. Carga del dataset

Cargamos las variables generadas por `2_feature_engineering.ipynb`.

No modificamos aquí las variables. La preparación de características pertenece a la etapa anterior.

In [19]:
# ==========================================
# 03. Carga del dataset
# ==========================================

data = pd.read_parquet(INPUT_PATH)

print("Dimensiones:", data.shape)
display(data.head())

Dimensiones: (73143, 42)


,machineID,dt_truncated,volt_rollingmean_12,volt_rollingstd_12,rotate_rollingmean_12,rotate_rollingstd_12,pressure_rollingmean_12,pressure_rollingstd_12,vibration_rollingmean_12,vibration_rollingstd_12,...,comp2sum,comp3sum,comp4sum,age,model_model1,model_model2,model_model3,model_model4,failure,label_e
0,1,2015-01-01 00:00:00,169.512597,6.239068,425.853115,46.581966,101.886267,13.166950,41.119335,4.473233,...,213.0,153.0,168.0,18,0,0,1,0,0.0,0.0
1,1,2015-01-01 12:00:00,167.007032,8.260186,434.948972,50.000568,100.970047,10.863588,39.412711,5.716674,...,214.0,154.0,169.0,18,0,0,1,0,0.0,4.0
2,1,2015-01-02 00:00:00,171.754734,13.479925,454.534311,44.925868,93.989345,7.963589,40.265190,5.761581,...,214.0,154.0,169.0,18,0,0,1,0,0.0,4.0
3,1,2015-01-02 12:00:00,168.184995,15.356860,444.596343,39.260749,100.778017,10.107193,39.209928,5.957205,...,215.0,155.0,170.0,18,0,0,1,0,0.0,4.0
4,1,2015-01-03 00:00:00,172.717240,12.879603,468.079805,40.727721,99.586915,12.005493,40.406556,5.574350,...,215.0,155.0,170.0,18,0,0,1,0,0.0,4.0


## 04. Separación X / y

`label_e` es la variable objetivo.

Las columnas identificadoras, temporales y utilizadas para construir la etiqueta no forman parte de las variables predictoras.

La lista de columnas debe mantenerse consistente con el Notebook 3 para evitar diferencias entre entrenamiento y producción.

In [20]:
# ==========================================
# 04. X / y
# ==========================================

input_features = [
    col for col in data.columns
    if col not in REMOVE_COLUMNS
]

X = data[input_features].copy()
y = data[TARGET].copy()

print("Variable objetivo:", TARGET)
print("Variables predictoras:", len(input_features))
print("X:", X.shape)
print("y:", y.shape)

Variable objetivo: label_e
Variables predictoras: 38
X: (73143, 38)
y: (73143,)


## 05. Train / Test

Separamos:

- **70 % Train:** tuning, Cross Validation y entrenamiento.
- **30 % Test:** evaluación final.

El Test Set no participa en `RandomizedSearchCV`. Esto evita optimizar indirectamente el modelo sobre los datos que utilizaremos para medir su generalización.

In [21]:
# ==========================================
# 05. Train / Test
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (51200, 38)
Test : (21943, 38)


## 06. Diagnóstico de imbalance

El Notebook 3 mostró un fuerte desbalance de clases. Por eso mantenemos SMOTE durante el entrenamiento.

### Regla crítica

SMOTE **no se aplica antes de Cross Validation**.

El Pipeline ejecutará:

```text
Fold de entrenamiento
      ↓
SMOTE
      ↓
Random Forest
      ↓
Validación del fold
```

De esta manera, los datos sintéticos se generan únicamente a partir del subconjunto de entrenamiento de cada fold.

> El Test Set nunca recibe SMOTE.

In [22]:
# ==========================================
# 06. Diagnóstico de imbalance
# ==========================================

class_counts = y_train.value_counts().sort_index()
display(class_counts.to_frame("Count"))

print("Porcentaje por clase:")
display((class_counts / len(y_train) * 100).round(2).to_frame("Percentage"))

,Count
0.0,47218
1.0,942
2.0,1305
3.0,700
4.0,1035


Porcentaje por clase:


,Percentage
0.0,92.22
1.0,1.84
2.0,2.55
3.0,1.37
4.0,2.02


## 07. Pipeline de entrenamiento

Construimos el Pipeline que será utilizado dentro de la búsqueda de hiperparámetros.

Esto es especialmente importante porque el Pipeline encapsula el proceso de entrenamiento y evita **data leakage**.

El modelo base conserva los hiperparámetros generales del Notebook 3 y la búsqueda modificará únicamente los parámetros definidos en el espacio de búsqueda.

In [23]:
# ==========================================
# 07. Pipeline
# ==========================================

pipeline = Pipeline([
    (
        "smote",
        SMOTE(
            random_state=RANDOM_STATE
        )
    ),
    (
        "classifier",
        RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )
])

print(pipeline)

Pipeline(steps=[('smote', SMOTE(random_state=42)),
                ('classifier',
                 RandomForestClassifier(n_jobs=-1, random_state=42))])


## 08. Espacio de hiperparámetros

Los hiperparámetros controlan cómo aprende Random Forest.

### Parámetros principales

- `n_estimators`: número de árboles.
- `max_depth`: profundidad máxima.
- `min_samples_split`: mínimo de observaciones para dividir un nodo.
- `min_samples_leaf`: mínimo de observaciones en una hoja.
- `max_features`: número de variables consideradas en cada división.
- `class_weight`: alternativa para tratar desbalance mediante pesos.

En este laboratorio utilizamos SMOTE como estrategia principal de balanceo, por lo que mantenemos `class_weight=None`.

### Relación con overfitting

Reducir la complejidad mediante `max_depth`, `min_samples_leaf` y `min_samples_split` puede ayudar a disminuir el sobreajuste.

In [24]:
# ==========================================
# 08. Espacio de búsqueda
# ==========================================

# Búsqueda compacta para completar el End-to-End rápidamente.
# Priorizamos hiperparámetros que controlan:
# - Complejidad del bosque: n_estimators / max_depth
# - Sobreajuste: min_samples_split / min_samples_leaf
# - Diversidad de variables: max_features

param_distributions = {
    "classifier__n_estimators": [50, 100],
    "classifier__max_depth": [8, 12, None],
    "classifier__min_samples_split": [2, 10],
    "classifier__min_samples_leaf": [1, 5, 10],
    "classifier__max_features": ["sqrt", "log2"]
}

print("Hiperparámetros configurados para búsqueda rápida:")
for parameter, values in param_distributions.items():
    print(f"- {parameter}: {values}")

print(f"CV folds: {N_SPLITS_TUNING}")
print(f"Iteraciones aleatorias: {N_ITER_SEARCH}")
print(f"Fits máximos: {N_SPLITS_TUNING * N_ITER_SEARCH}")


Hiperparámetros configurados para búsqueda rápida:
- classifier__n_estimators: [50, 100]
- classifier__max_depth: [8, 12, None]
- classifier__min_samples_split: [2, 10]
- classifier__min_samples_leaf: [1, 5, 10]
- classifier__max_features: ['sqrt', 'log2']
CV folds: 2
Iteraciones aleatorias: 3
Fits máximos: 6


## 09. Randomized Search + Cross Validation

La búsqueda se ejecuta **solo sobre Train**.

Cada configuración se evalúa mediante Stratified K-Fold y se optimiza usando **F1 Macro**, porque las clases están fuertemente desbalanceadas y queremos dar el mismo peso a cada clase.

```text
TRAIN
  │
  ├── Fold 1 → SMOTE → RF → validación
  ├── Fold 2 → SMOTE → RF → validación
  └── Fold 3 → SMOTE → RF → validación
              ↓
         F1 Macro medio
              ↓
       mejor configuración
```

El Test Set continúa aislado.

In [25]:
# ==========================================
# 09. RandomizedSearchCV
# ==========================================

# Usamos pocos folds e iteraciones para que el laboratorio sea ejecutable
# en un Compute pequeño. El objetivo es demostrar el proceso profesional,
# no realizar una búsqueda exhaustiva de producción.

cv_tuning = StratifiedKFold(
    n_splits=N_SPLITS_TUNING,
    shuffle=True,
    random_state=RANDOM_STATE
)

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=N_ITER_SEARCH,
    scoring="f1_macro",
    cv=cv_tuning,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
    refit=True
)

print(
    f"Ejecutando {N_ITER_SEARCH} configuraciones x "
    f"{N_SPLITS_TUNING} folds = "
    f"{N_ITER_SEARCH * N_SPLITS_TUNING} fits."
)

start_time = time.time()

search.fit(X_train, y_train)

elapsed = time.time() - start_time

print(f"Tiempo de búsqueda: {elapsed/60:.2f} minutos")
print("Mejor F1 Macro CV:", round(search.best_score_, 4))
print("Mejores hiperparámetros:")

for k, v in search.best_params_.items():
    print(f"  {k}: {v}")


Ejecutando 3 configuraciones x 2 folds = 6 fits.
Fitting 2 folds for each of 3 candidates, totalling 6 fits
Tiempo de búsqueda: 3.77 minutos
Mejor F1 Macro CV: 0.615
Mejores hiperparámetros:
  classifier__n_estimators: 50
  classifier__min_samples_split: 10
  classifier__min_samples_leaf: 1
  classifier__max_features: log2
  classifier__max_depth: None


## 10. Resultados del tuning

Revisamos las configuraciones exploradas y ordenamos por F1 Macro de validación.

Además de la métrica de validación, conservamos el resultado de entrenamiento para identificar configuraciones que podrían estar sobreajustándose.

In [26]:
# ==========================================
# 10. Resultados del tuning
# ==========================================

cv_results = pd.DataFrame(search.cv_results_)

tuning_results = cv_results[
    [
        "rank_test_score",
        "mean_train_score",
        "std_train_score",
        "mean_test_score",
        "std_test_score",
        "params"
    ]
].sort_values("rank_test_score")

display(tuning_results.round(4))

,rank_test_score,mean_train_score,std_train_score,mean_test_score,std_test_score,params
1,1,0.9919,0.0011,0.6150,0.0017,"{'classifier__n_estimators': 50, 'classifier__..."
0,2,0.5029,0.0015,0.4729,0.0037,"{'classifier__n_estimators': 50, 'classifier__..."
2,3,0.4771,0.0075,0.4502,0.0031,"{'classifier__n_estimators': 50, 'classifier__..."


## 11. Mejor modelo y análisis de sobreajuste

`RandomizedSearchCV` ya reentrenó el mejor Pipeline sobre todo el conjunto Train.

Calculamos:

- F1 Macro en Train.
- F1 Macro en Test.
- Brecha Train–Test.

Una brecha grande indica que el modelo aprende patrones específicos del entrenamiento que no se mantienen completamente sobre datos no vistos.

> Una reducción de la brecha es deseable, pero no debe conseguirse sacrificando excesivamente el desempeño.

In [27]:
# ==========================================
# 11. Mejor modelo + overfitting
# ==========================================

tuned_model = search.best_estimator_

train_pred = tuned_model.predict(X_train)
test_pred = tuned_model.predict(X_test)

train_f1_macro = f1_score(
    y_train,
    train_pred,
    average="macro"
)

test_f1_macro = f1_score(
    y_test,
    test_pred,
    average="macro"
)

train_test_gap = train_f1_macro - test_f1_macro

overfitting_status = (
    "Posible overfitting"
    if train_test_gap >= 0.10
    else "Sin señal fuerte"
)

summary = pd.DataFrame([{
    "Model": "Tuned Random Forest",
    "Best CV F1 Macro": search.best_score_,
    "Train F1 Macro": train_f1_macro,
    "Test F1 Macro": test_f1_macro,
    "Train-Test Gap": train_test_gap,
    "Status": overfitting_status
}])

display(summary.round(4))

,Model,Best CV F1 Macro,Train F1 Macro,Test F1 Macro,Train-Test Gap,Status
0,Tuned Random Forest,0.615,0.9903,0.6149,0.3755,Posible overfitting


## 12. Evaluación final sobre Test

El Test Set se utiliza ahora, por primera vez en este notebook, para obtener una estimación final del desempeño del modelo optimizado.

Esta evaluación **no modifica los hiperparámetros**.

Calculamos las métricas utilizadas durante el proyecto:

- Accuracy
- Precision Macro
- Recall Macro
- F1 Macro
- ROC-AUC OVR Macro

F1 Macro continúa siendo la métrica principal.

In [28]:
# ==========================================
# 12. Evaluación final
# ==========================================

test_prob = tuned_model.predict_proba(X_test)

test_metrics = {
    "Accuracy": accuracy_score(y_test, test_pred),
    "Precision Macro": precision_score(
        y_test, test_pred, average="macro", zero_division=0
    ),
    "Recall Macro": recall_score(
        y_test, test_pred, average="macro", zero_division=0
    ),
    "F1 Macro": f1_score(
        y_test, test_pred, average="macro", zero_division=0
    ),
    "ROC-AUC OVR Macro": roc_auc_score(
        y_test,
        test_prob,
        multi_class="ovr",
        average="macro"
    )
}

metrics_df = pd.DataFrame(
    [test_metrics]
)

display(metrics_df.round(4))

,Accuracy,Precision Macro,Recall Macro,F1 Macro,ROC-AUC OVR Macro
0,0.9417,0.7265,0.5449,0.6149,0.9119


## 13. Classification Report

El F1 Macro resume el desempeño de todas las clases con el mismo peso, pero necesitamos revisar el comportamiento de cada clase.

Esto permite detectar si el modelo funciona bien para la clase mayoritaria pero continúa teniendo dificultades con alguna clase minoritaria.

In [29]:
# ==========================================
# 13. Classification Report
# ==========================================

report = classification_report(
    y_test,
    test_pred,
    zero_division=0,
    output_dict=True
)

report_df = pd.DataFrame(report).T

display(report_df.round(4))

,precision,recall,f1-score,support
0.0,0.9557,0.9846,0.9699,20236.0000
1.0,0.5655,0.3738,0.4501,404.0000
2.0,0.7732,0.4321,0.5544,560.0000
3.0,0.6605,0.4733,0.5515,300.0000
4.0,0.6777,0.4605,0.5484,443.0000
accuracy,0.9417,0.9417,0.9417,0.9417
macro avg,0.7265,0.5449,0.6149,21943.0000
weighted avg,0.9342,0.9417,0.9355,21943.0000


## 14. Confirmación mediante Cross Validation

Después de seleccionar los hiperparámetros, realizamos una validación adicional sobre Train.

Esta etapa permite comprobar que el resultado del modelo optimizado no depende de una única partición.

El Test Set continúa fuera de esta Cross Validation.

In [30]:
# ==========================================
# 14. Cross Validation de confirmación
# ==========================================

cv_confirmation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro"
}

confirmation = cross_validate(
    tuned_model,
    X_train,
    y_train,
    cv=cv_confirmation,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

confirmation_df = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision Macro",
        "Recall Macro",
        "F1 Macro"
    ],
    "Train Mean": [
        confirmation["train_accuracy"].mean(),
        confirmation["train_precision_macro"].mean(),
        confirmation["train_recall_macro"].mean(),
        confirmation["train_f1_macro"].mean()
    ],
    "CV Mean": [
        confirmation["test_accuracy"].mean(),
        confirmation["test_precision_macro"].mean(),
        confirmation["test_recall_macro"].mean(),
        confirmation["test_f1_macro"].mean()
    ]
})

confirmation_df["Train-CV Gap"] = (
    confirmation_df["Train Mean"] -
    confirmation_df["CV Mean"]
)

display(confirmation_df.round(4))

/anaconda/envs/azureml_py38/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:700: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


## 15. Decisión del modelo optimizado

La decisión no se basa exclusivamente en la métrica de Train.

Consideramos conjuntamente:

1. **F1 Macro en Cross Validation**.
2. **F1 Macro en Test**.
3. Brecha Train–CV.
4. Brecha Train–Test.
5. Desempeño por clase.
6. Complejidad del modelo.

Si el modelo optimizado mejora la generalización sin deteriorar significativamente el desempeño, será el candidato para las siguientes etapas.

> Para completar el laboratorio End-to-End, este modelo será el artefacto que pasará a **Model Serialization → Deployment → Endpoint → Batch Inference → Monitoring → Power BI**.

In [ ]:
# ==========================================
# 15. Resumen de decisión
# ==========================================

cv_f1_macro = confirmation[
    "test_f1_macro"
].mean()

cv_f1_std = confirmation[
    "test_f1_macro"
].std()

print("==========================================")
print("MODELO OPTIMIZADO")
print("==========================================")
print("Random Forest")
print(f"CV F1 Macro : {cv_f1_macro:.4f}")
print(f"CV Std      : {cv_f1_std:.4f}")
print(f"Test F1 Macro: {test_metrics['F1 Macro']:.4f}")
print(f"Train-Test Gap: {train_test_gap:.4f}")
print(f"Estado: {overfitting_status}")

## 16. Persistencia de artefactos

Guardamos el Pipeline completo y los resultados necesarios para las siguientes etapas.

El objeto `tuned_random_forest.pkl` contiene:

```text
SMOTE
  +
Random Forest
  +
hiperparámetros optimizados
```

Esto permite que el mismo proceso de transformación aprendido durante entrenamiento se reproduzca cuando el modelo sea utilizado para inferencia.

También guardamos:

- resultados de la búsqueda;
- métricas finales;
- análisis de generalización;
- hiperparámetros seleccionados.

Estos artefactos permiten trazabilidad y facilitan la transición hacia producción.

In [ ]:
# ==========================================
# 16. Guardar artefactos
# ==========================================

TUNED_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "tuned_random_forest.pkl"
)

TUNING_RESULTS_PATH = os.path.join(
    MODEL_DIR,
    "random_forest_tuning_results.csv"
)

METRICS_PATH = os.path.join(
    MODEL_DIR,
    "tuned_random_forest_metrics.csv"
)

CONFIRMATION_PATH = os.path.join(
    MODEL_DIR,
    "tuned_random_forest_cv_confirmation.csv"
)

joblib.dump(
    tuned_model,
    TUNED_MODEL_PATH
)

tuning_results.to_csv(
    TUNING_RESULTS_PATH,
    index=False
)

summary.to_csv(
    METRICS_PATH,
    index=False
)

confirmation_df.to_csv(
    CONFIRMATION_PATH,
    index=False
)

print("Artefactos guardados:")
print(TUNED_MODEL_PATH)
print(TUNING_RESULTS_PATH)
print(METRICS_PATH)
print(CONFIRMATION_PATH)

# Resumen del Notebook 4

Al finalizar esta etapa tendremos:

```text
✓ Random Forest seleccionado como candidato
✓ Hiperparámetros optimizados
✓ RandomizedSearchCV compacto
✓ SMOTE dentro del Pipeline
✓ Test Set aislado
✓ Cross Validation de confirmación
✓ Evaluación final sobre Test
✓ Análisis de overfitting
✓ Classification Report
✓ Modelo optimizado serializado
✓ Métricas y resultados guardados
```

## Siguiente etapa

El modelo optimizado pasa a:

```text
04 Hyperparameter Tuning
          ↓
05 Model Evaluation
          ↓
06 Model Serialization
          ↓
07 Deployment
          ↓
08 Batch Inference
          ↓
09 Monitoring
          ↓
Power BI / Aplicaciones
```

**Importante:** si posteriormente Gradient Boosting u otro modelo supera a Random Forest después del tuning, se puede sustituir el artefacto sin cambiar la arquitectura de inferencia y consumo.
## Nota de laboratorio

La búsqueda se mantiene deliberadamente pequeña para priorizar la demostración
End-to-End. En un proyecto productivo real se pueden aumentar los folds,
iteraciones y espacio de búsqueda, idealmente utilizando un proceso de
experiment tracking y un presupuesto computacional definido.
